In [0]:

sc = spark.sparkContext  


# Linear SVC to predict whether an incident met SLA

The Incident Management dataset has about 141712 records of 24918 incidents. Each state of the incident is being captured as an individual record with few exceptions where the closed state of an incident is recorded more than once. With the help of the below segment of the code, we load and clean the Incident Management data so that only one record representing the truly closed state per incident is obtained.

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType

df = spark.read.csv('/FileStore/tables/incident_event_log_reduced.csv', header=True, inferSchema=True)

# display the first 5 rows of the dataframe
df.show(5)

+----------+--------------+------+------------------+------------+-------------+--------+-----------+--------------+---------------+--------------+---------------+--------------+--------------+------------+------------+-----------+---------------+-----------+-------+----------+----------+------------+----------------+------------+---------+-----------------------+-------------+-------------+---+------+---------+-----------+---------------+---------------+--------------+
|    number|incident_state|active|reassignment_count|reopen_count|sys_mod_count|made_sla|  caller_id|     opened_by|      opened_at|sys_created_by| sys_created_at|sys_updated_by|sys_updated_at|contact_type|    location|   category|    subcategory|  u_symptom|cmdb_ci|    impact|   urgency|    priority|assignment_group| assigned_to|knowledge|u_priority_confirmation|       notify|   problem_id|rfc|vendor|caused_by|closed_code|    resolved_by|    resolved_at|     closed_at|
+----------+--------------+------+----------------

In [0]:
from pyspark.sql.functions import datediff,date_format,to_date,to_timestamp
import pyspark.sql.functions as f


spark.conf.set("spark.sql.legacy.timeParserPolicy","LEGACY")
 
df=df.withColumn('resolved_ts',to_timestamp(df.resolved_at, 'dd/MM/yyyy HH:mm')).\
        withColumn('opened_ts',to_timestamp(df.opened_at, 'dd/MM/yyyy HH:mm')).\
        withColumn('closed_ts',to_timestamp(df.closed_at, 'dd/MM/yyyy HH:mm')).\
        withColumn('resolved',to_date(df.resolved_at, 'dd/MM/yyyy HH:mm')).\
        withColumn('opened',to_date(df.opened_at, 'dd/MM/yyyy HH:mm')).\
        withColumn('closed',to_date(df.closed_at, 'dd/MM/yyyy HH:mm')).\
        withColumn('knowledge', f.col('knowledge').cast('string')).\
        replace(['TRUE',], 'True', subset='knowledge').\
        replace(['FALSE'], 'False', subset='knowledge').\
        withColumn('resolved_duration',datediff(to_date(df.resolved_at, 'dd/MM/yyyy HH:mm'),\
                                                to_date(df.opened_at, 'dd/MM/yyyy HH:mm'))).\
        withColumn('closed_duration',datediff(to_date(df.closed_at, 'dd/MM/yyyy HH:mm'),\
                                                to_date(df.opened_at, 'dd/MM/yyyy HH:mm'))).\
        withColumn('made_sla_int',df.made_sla.cast('integer'))

The data set has multiple states(New, Active, Awaiting user info, Resolved, Closed etc. ) of an incident. With the help of the below command, we are just filtering one record per incident, that has the truly closed state of the incident. 

In [0]:
df_unique_incidents=df.filter("incident_state=='Closed'").sort("sys_mod_count",ascending=False).dropDuplicates(["number"])

Selecting the dependent and the independent variables that are identified as most useful attributes to make predictions

In [0]:
data=df_unique_incidents.select([
    'sys_mod_count',
    'opened_by',
    'location',
    'category',
    'priority',
    'assignment_group',
    'knowledge',
    'resolved_duration',
    'closed_duration',
    'made_sla_int'
    ]
)

data=data.dropna()

Create a 70-30 train test split

In [0]:
train_data,test_data=data.randomSplit([0.7,0.3])

### Building the Linear SVC model

In [0]:
from pyspark.ml.classification import LinearSVC
from pyspark.ml.feature import VectorAssembler,StringIndexer,StandardScaler
from pyspark.ml import Pipeline

Use StringIndexer to convert the categorical columns to hold numerical data

In [0]:
opened_by_indexer = StringIndexer(inputCol='opened_by',outputCol='opened_by_index',handleInvalid='keep')
location_indexer = StringIndexer(inputCol='location',outputCol='location_index',handleInvalid='keep')
category_indexer = StringIndexer(inputCol='category',outputCol='category_index',handleInvalid='keep')
priority_indexer = StringIndexer(inputCol='priority',outputCol='priority_index',handleInvalid='keep')
assignment_group_indexer = StringIndexer(inputCol='assignment_group',outputCol='assignment_group_index',handleInvalid='keep')
knowledge_indexer = StringIndexer(inputCol='knowledge',outputCol='knowledge_index',handleInvalid='keep')

Vector assembler is used to create a vector of input features.

In [0]:
assembler = VectorAssembler(
    inputCols=[
        'opened_by_index',
        'location_index',
        'category_index',
        'priority_index',
        'assignment_group_index',
        'knowledge_index'
    ],
    outputCol="unscaled_features"
)

Standard scaler is used to scale the data for the linear SVC to perform well on the training data

In [0]:
scaler = StandardScaler(inputCol="unscaled_features",outputCol="features")

Create an object for the Linear SVC model

In [0]:
svc_model = LinearSVC(labelCol='made_sla_int')

Pipeline is used to pass the data through indexer and assembler simultaneously. Also, it helps to pre-rocess the test data in the same way as that of the train data. 

In [0]:
pipe = Pipeline(
    stages=[
        opened_by_indexer,
        location_indexer,
        category_indexer,
        priority_indexer,
        assignment_group_indexer,
        knowledge_indexer,
        assembler,scaler,svc_model
    ]
)

Fit the model on the training data

In [0]:
train_data.printSchema()

root
 |-- sys_mod_count: integer (nullable = true)
 |-- opened_by: string (nullable = true)
 |-- location: string (nullable = true)
 |-- category: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- assignment_group: string (nullable = true)
 |-- knowledge: string (nullable = true)
 |-- resolved_duration: integer (nullable = true)
 |-- closed_duration: integer (nullable = true)
 |-- made_sla_int: integer (nullable = true)



In [0]:
test_data.printSchema()

root
 |-- sys_mod_count: integer (nullable = true)
 |-- opened_by: string (nullable = true)
 |-- location: string (nullable = true)
 |-- category: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- assignment_group: string (nullable = true)
 |-- knowledge: string (nullable = true)
 |-- resolved_duration: integer (nullable = true)
 |-- closed_duration: integer (nullable = true)
 |-- made_sla_int: integer (nullable = true)



In [0]:
%time

fit_model=pipe.fit(train_data)

CPU times: user 5 µs, sys: 1 µs, total: 6 µs
Wall time: 11 µs


Store the results in a dataframe

In [0]:
results = fit_model.transform(test_data)
display(results)

sys_mod_count,opened_by,location,category,priority,assignment_group,knowledge,resolved_duration,closed_duration,made_sla_int,opened_by_index,location_index,category_index,priority_index,assignment_group_index,knowledge_index,unscaled_features,features,rawPrediction,prediction
1,?,Location 111,Category 32,3 - Moderate,?,false,0,5,1,8.0,9.0,4.0,0.0,1.0,0.0,"Map(vectorType -> dense, length -> 6, values -> List(8.0, 9.0, 4.0, 0.0, 1.0, 0.0))","Map(vectorType -> dense, length -> 6, values -> List(0.41960138442977746, 0.43720495508471036, 0.6326696819576296, 0.0, 0.0878334612548565, 0.0))","Map(vectorType -> dense, length -> 2, values -> List(-1.000000008957739, 1.000000008957739))",1.0
1,?,Location 143,Category 32,3 - Moderate,?,false,0,5,1,8.0,2.0,4.0,0.0,1.0,0.0,"Map(vectorType -> dense, length -> 6, values -> List(8.0, 2.0, 4.0, 0.0, 1.0, 0.0))","Map(vectorType -> dense, length -> 6, values -> List(0.41960138442977746, 0.09715665668549119, 0.6326696819576296, 0.0, 0.0878334612548565, 0.0))","Map(vectorType -> dense, length -> 2, values -> List(-1.000000008957739, 1.000000008957739))",1.0
1,?,Location 15,Category 32,3 - Moderate,?,false,0,5,1,8.0,27.0,4.0,0.0,1.0,0.0,"Map(vectorType -> dense, length -> 6, values -> List(8.0, 27.0, 4.0, 0.0, 1.0, 0.0))","Map(vectorType -> dense, length -> 6, values -> List(0.41960138442977746, 1.311614865254131, 0.6326696819576296, 0.0, 0.0878334612548565, 0.0))","Map(vectorType -> dense, length -> 2, values -> List(-1.000000008957739, 1.000000008957739))",1.0
1,?,Location 204,Category 23,3 - Moderate,Group 20,false,0,63,0,8.0,0.0,7.0,0.0,10.0,0.0,"Map(vectorType -> dense, length -> 6, values -> List(8.0, 0.0, 7.0, 0.0, 10.0, 0.0))","Map(vectorType -> dense, length -> 6, values -> List(0.41960138442977746, 0.0, 1.1071719434258518, 0.0, 0.8783346125485649, 0.0))","Map(vectorType -> dense, length -> 2, values -> List(-0.9999999662138744, 0.9999999662138744))",1.0
1,?,Location 204,Category 32,3 - Moderate,?,false,0,5,1,8.0,0.0,4.0,0.0,1.0,0.0,"Map(vectorType -> dense, length -> 6, values -> List(8.0, 0.0, 4.0, 0.0, 1.0, 0.0))","Map(vectorType -> dense, length -> 6, values -> List(0.41960138442977746, 0.0, 0.6326696819576296, 0.0, 0.0878334612548565, 0.0))","Map(vectorType -> dense, length -> 2, values -> List(-1.000000008957739, 1.000000008957739))",1.0
1,?,Location 51,Category 32,3 - Moderate,?,false,0,5,1,8.0,5.0,4.0,0.0,1.0,0.0,"Map(vectorType -> dense, length -> 6, values -> List(8.0, 5.0, 4.0, 0.0, 1.0, 0.0))","Map(vectorType -> dense, length -> 6, values -> List(0.41960138442977746, 0.24289164171372796, 0.6326696819576296, 0.0, 0.0878334612548565, 0.0))","Map(vectorType -> dense, length -> 2, values -> List(-1.000000008957739, 1.000000008957739))",1.0
1,?,Location 51,Category 32,3 - Moderate,?,false,0,5,1,8.0,5.0,4.0,0.0,1.0,0.0,"Map(vectorType -> dense, length -> 6, values -> List(8.0, 5.0, 4.0, 0.0, 1.0, 0.0))","Map(vectorType -> dense, length -> 6, values -> List(0.41960138442977746, 0.24289164171372796, 0.6326696819576296, 0.0, 0.0878334612548565, 0.0))","Map(vectorType -> dense, length -> 2, values -> List(-1.000000008957739, 1.000000008957739))",1.0
1,Opened by 168,Location 161,Category 24,3 - Moderate,Group 24,false,0,5,1,33.0,1.0,11.0,0.0,4.0,0.0,"Map(vectorType -> dense, length -> 6, values -> List(33.0, 1.0, 11.0, 0.0, 4.0, 0.0))","Map(vectorType -> dense, length -> 6, values -> List(1.730855710772832, 0.04857832834274559, 1.7398416253834814, 0.0, 0.351333845019426, 0.0))","Map(vectorType -> dense, length -> 2, values -> List(-1.0000000099273574, 1.0000000099273574))",1.0
1,Opened by 168,Location 161,Category 24,3 - Moderate,Group 24,false,0,6,1,33.0,1.0,11.0,0.0,4.0,0.0,"Map(vectorType -> dense, length -> 6, values -> List(33.0, 1.0, 11.0, 0.0, 4.0, 0.0))","Map(vectorType -> dense, length -> 6, values -> List(1.730855710772832, 0.04857832834274559, 1.7398416253834814, 0.0, 0.351333845019426, 0.0))","Map(vectorType -> dense, length -> 2, values -> List(-1.0000000099273574, 1.000000009

In [0]:
results.select(['made_sla_int','prediction']).show()

+------------+----------+
|made_sla_int|prediction|
+------------+----------+
|           1|       1.0|
|           1|       1.0|
|           1|       1.0|
|           0|       1.0|
|           1|       1.0|
|           1|       1.0|
|           1|       1.0|
|           1|       1.0|
|           1|       1.0|
|           1|       1.0|
|           1|       1.0|
|           1|       1.0|
|           1|       1.0|
|           1|       1.0|
|           0|       1.0|
|           0|       1.0|
|           0|       1.0|
|           0|       1.0|
|           0|       1.0|
|           0|       1.0|
+------------+----------+
only showing top 20 rows



### Evaluating the model

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

AUC_evaluator = BinaryClassificationEvaluator(rawPredictionCol='prediction',labelCol='made_sla_int',metricName='areaUnderROC')
AUC = AUC_evaluator.evaluate(results)

print(f"The area under the curve is {AUC:.2f}")

The area under the curve is 0.65


A roughly 65% area under ROC denotes the model has performed reasonably well in predicting whether an incident has met the sla

###  Area under the PR curve

In [0]:
PR_evaluator = BinaryClassificationEvaluator(rawPredictionCol='prediction',labelCol='made_sla_int',metricName='areaUnderPR')
PR = PR_evaluator.evaluate(results)

print("The area under the PR curve is {}".format(PR))

The area under the PR curve is 0.6973831700678683


### Accuracy

In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

ACC_evaluator = MulticlassClassificationEvaluator(
    labelCol="made_sla_int", predictionCol="prediction", metricName="accuracy")

accuracy = ACC_evaluator.evaluate(results)

print("The accuracy of the model is {}".format(accuracy))

The accuracy of the model is 0.7146334478808706


### Confusion Matrix

In [0]:
from sklearn.metrics import confusion_matrix

y_true = results.select("made_sla_int")
y_true = y_true.toPandas()
 
y_pred = results.select("prediction")
y_pred = y_pred.toPandas()
 
cnf_matrix = confusion_matrix(y_true, y_pred)
print("Below is the confusion matrix: \n {}".format(cnf_matrix))

Below is the confusion matrix: 
 [[ 955 1730]
 [ 263 4036]]


In [0]:
tn = cnf_matrix[0][0]
fp = cnf_matrix[0][1]
fn = cnf_matrix[1][0]
tp = cnf_matrix[1][1]

accuracy = (tp+tn)/(tp+tn+fp+fn)
precision = tp/(tp+fp)
recall = tp/(tp+fn)
f1_score = 2*(precision*recall)/(precision+recall)

In [0]:
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1_score:.2f}")

Accuracy: 0.71
Precision: 0.70
Recall: 0.94
F1 Score: 0.80


In [0]:
spark.stop()